In [ ]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

In [ ]:
using LinearAlgebra
using ForwardDiff
using PyPlot

In [ ]:
Q = Diagonal([0.5; 1])
function f(x)
    return 0.5*(x-[1; 0])'*Q*(x-[1; 0])
end
function ∇f(x)
    return Q*(x-[1; 0])
end
function ∇2f(x)
    return Q
end

In [ ]:
A = [-1.0 1.0]
b = 1.0
function c(x)
    return dot(A,x) - b
end
function ∂c(x)
    return A
end
# -x+y - 1 < 0

In [ ]:
function plot_landscape()
    Nsamp = 20
    Xsamp = kron(ones(Nsamp),LinRange(-4,4,Nsamp)')
    Ysamp = kron(ones(Nsamp)',LinRange(-4,4,Nsamp))
    Zsamp = zeros(Nsamp,Nsamp)
    for j = 1:Nsamp
        for k = 1:Nsamp
            Zsamp[j,k] = f([Xsamp[j,k]; Ysamp[j,k]])
        end
    end
    contour(Xsamp,Ysamp,Zsamp)

    xc = LinRange(-4,3,Nsamp)
    plot(xc,xc.+1,"y")
end

plot_landscape()

In [ ]:
function ip_residual_4var(z, ρ)
    # 1. 변수 4개 분할 (z는 이제 4차원 벡터입니다)
    x = z[1:2]  # 원 변수 (Primal variables: x1, x2)
    s = z[3]    # 여유 변수 (Slack variable)
    μ = z[4]    # 라그랑주 승수 (Dual variable)
    
    # 2. 완화된 KKT 시스템 방정식 (슬라이드 12페이지)
    # ∇f(x) - ∂c(x)' * μ = 0 (목적함수와 제약조건의 기울기 균형)
    r_x = ∇f(x) - ∂c(x)' * μ
    
    # c(x) - s = 0 (여유 변수를 이용해 부등식을 등식으로 만듦)
    r_s = c(x) - s
    
    # s * μ = ρ (완화된 상보성 조건, Relaxed Complementarity)
    r_μ = s * μ - ρ
    
    # 3. 전체 잔차(Residual) 벡터로 묶어서 반환 (총 4개 요소)
    return [r_x; r_s; r_μ]
end

In [ ]:
function kkt_residual_4var(z)
    # 1. 4차원 벡터에서 변수 추출
    x = z[1:2]  # 원 변수 (x1, x2)
    s = z[3]    # 여유 변수 (Slack variable)
    μ = z[4]    # 라그랑주 승수 (Dual variable, 기존 코드의 λ)

    # 2. 오리지널 KKT 조건 검사 (슬라이드 11페이지 기반)
    r = [∇f(x) - ∂c(x)' * μ;  # (1) Stationarity: 기울기 균형
         c(x) - s;            # (2) Primal Equality: c(x)와 s는 같아야 함
         s * μ;               # (3) Complementary Slackness: 진짜 KKT이므로 ρ가 아니라 0이어야 함!
         min(μ, 0);           # (4) Dual Feasibility: μ >= 0 이어야 함 (음수면 오차 발생)
         min(s, 0)]           # (5) Primal Feasibility: s >= 0 이어야 함 (음수면 오차 발생)
         
    return r
end

In [ ]:
xguess = [-2; 2]
σguess = 0.0
z = [xguess; σguess]
plot_landscape()
plot(z[1], z[2], "rx")

In [ ]:
function newton_solve_4var(z0, ρ, tol)
    z = copy(z0)
    
    # 4변수 잔차 함수 사용
    r = ip_residual_4var(z, ρ)

    while norm(r) > tol       
        # M은 이제 4x4 야코비안 행렬이 됩니다!
        M = ForwardDiff.jacobian(dz -> ip_residual_4var(dz, ρ), z)
        
        # 뉴턴 스텝 (방향과 거리) 계산
        Δz = -M \ r

        # --- [추가된 핵심 안전장치: Fraction-to-the-boundary rule] ---
        # s와 μ가 0 이하로 떨어지는 것을 막기 위해 최대 보폭(α_max)을 계산합니다.
        s = z[3];  Δs = Δz[3]
        μ = z[4];  Δμ = Δz[4]
        
        α_max = 1.0 # 기본 최대 보폭은 100%
        
        # 만약 s가 줄어드는 방향(Δs < 0)으로 이동한다면, 0에 닿기 직전(0.99)까지만 이동 허용
        if Δs < 0
            α_max = min(α_max, -0.99 * s / Δs)
        end
        # 만약 μ가 줄어드는 방향(Δμ < 0)으로 이동한다면, 0에 닿기 직전(0.99)까지만 이동 허용
        if Δμ < 0
            α_max = min(α_max, -0.99 * μ / Δμ)
        end
        # -------------------------------------------------------------

        # Line search (선형 탐색)
        b = 0.1
        c = 0.5
        α = α_max # 1.0 대신, 안전하게 계산된 α_max부터 시작!
        
        znew = z + α * Δz
        rnew = ip_residual_4var(znew, ρ)

        # 오차가 충분히 줄어들 때까지 보폭(α)을 반절씩 줄임
        while norm(rnew) > (norm(r) + b * α * dot(r, M * Δz) / norm(r))
            α = c * α
            znew = z + α * Δz
            rnew = ip_residual_4var(znew, ρ)
        end

        z = znew
        r = rnew
    end

    return z
end

In [ ]:
# 1. 초기 상태 설정
# z = [x1, x2, s, μ] 순서입니다.
# s와 μ는 반드시 양수(>0)로 시작해야 합니다!
z_current = [0.0, 2.0, 1.0, 1.0] 
ρ = 10

println("=== 최적화 시작 ===")
plot_landscape()
# 2. 방벽 파라미터 ρ를 점진적으로 줄여가며 진짜 최적해를 찾음 (Central Path)
for i in 1:10
    # 현재 ρ 값에 대해 KKT 잔차가 0이 될 때까지 뉴턴법 실행
    z_current = newton_solve_4var(z_current, ρ, 1e-5)
    
    # KKT_residual_4var로 진짜 정답(오차)에 얼마나 가까워졌는지 확인
    real_error = norm(kkt_residual_4var(z_current))
    
    println("Iteration $i | ρ = ", round(ρ, digits=5), 
            " | 현재 x = [", round(z_current[1], digits=4), ", ", round(z_current[2], digits=4), "]",
            " | 진짜 KKT 오차 = ", round(real_error, digits=6))
    
    # 방벽 강도(ρ)를 10분의 1로 줄임
    ρ = 0.1 * ρ
    plot(z_current[1], z_current[2], "rx")
end

println("=== 최적화 완료 ===")




In [ ]:
function ip_residual(z, ρ)
    x = z[1:2]
    σ = z[3]
    r = [∇f(x) - ∂c(x)'*sqrt(ρ)*exp(-σ);
         c(x) - sqrt(ρ)exp(σ)]
end

In [ ]:
function kkt_residual(z)
    x = z[1:2]
    σ = z[3]
    λ = sqrt(ρ)*exp(-σ)

    r = [∇f(x) - ∂c(x)'*λ;
         min(λ, 0)
         min(c(x),0)
         λ*c(x)]
end

In [ ]:
ρ = 1.0
ip_residual(z,ρ)

In [ ]:
kkt_residual(z)

In [ ]:
function newton_solve(z0,ρ,tol)

    #initial guess
    z = z0
    
    #KKT residual
    r = ip_residual(z,ρ)

    while norm(r) > tol       
        #H = ∇2f(x)
        #C = ∂c(x)

        #M = [H sqrt(ρ)*C'*exp(-σ);
        #    C -sqrt(ρ)*exp(σ)]

        #Newton step
        M = ForwardDiff.jacobian(dz->ip_residual(dz,ρ), z)
        Δz = -M\r

        znew = z + Δz
        rnew = ip_residual(znew,ρ)

        #Line search
        b = 0.1
        c = 0.5
        α = 1.0
        while norm(rnew) > (norm(r) + b*α*dot(r,M*Δz)/norm(r))
            α = c*α
            znew = z + α*Δz
            rnew = ip_residual(znew,ρ)
        end

        z = znew
        r = rnew
    end

    return z
end

In [ ]:
z_iter = z

In [ ]:
ρ = 1e-8#adjust from ρ=1 to ρ=1e-8 to observe convergence along central path
z = newton_solve(z_iter[:,end],ρ, 1e-10)
z_iter = [z_iter z]


In [ ]:
kkt_residual(z)

In [ ]:
plot_landscape()
plot(z_iter[1,:], z_iter[2,:], "rx")

In [ ]:
M = ForwardDiff.jacobian(dz->ip_residual(dz,ρ), z)

In [ ]:
eigvals(M)